# MTAP Revision Extras (Colab)

Yeh notebook **sirf revision extras** chalata hai:
1. Global-mean (**μ-only**) baseline — 6 Amazon category pairs
2. 5-fold CV, seed **42**, train fractions **20/40/60/80%** (same as paper notebooks)

**Kaise use karein**
1. Colab mein yeh updated notebook upload/open karein
2. Cell 1: Drive mount
3. Cell 2: auto-find / SNAP se `amazon-meta.txt` download (agar Drive pe nahi hai)
4. Baaki cells Run All
5. Last cell ka **pura printed output** (`COPY EVERYTHING...` se `END_OF_OUTPUT`) Cursor mein paste kar dein

**Note:** Aapke Drive pe pehle `amazon-meta.txt` nahi tha — isliye FileNotFound aaya tha. Cell 2 ab file khud download kar lega.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== Resolve amazon-meta.txt =====
# Paper uses SNAP amazon-meta (groups: Book/Music/DVD/Video).
# If missing on Drive, downloads from Stanford SNAP.

import os, gzip, shutil
from pathlib import Path

CANDIDATES = [
    "/content/drive/MyDrive/amazon-meta.txt",
    "/content/drive/MyDrive/amazon-meta.txt.gz",
    "/content/drive/MyDrive/Dataset-CDRS/amazon-meta.txt",
    "/content/drive/MyDrive/Enhance -Comprative(decomposition techniques)/amazon-meta.txt",
    "/content/amazon-meta.txt",
]

RATING_THRESHOLD = 30
N_FOLDS = 5
RANDOM_STATE = 42
FRACTIONS = (0.2, 0.4, 0.6, 0.8)

CATEGORY_PAIRS = [
    ("Book", "Music"),
    ("DVD", "Book"),
    ("Video", "Book"),
    ("DVD", "Music"),
    ("Video", "Music"),
    ("DVD", "Video"),
]

def find_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    root = Path("/content/drive/MyDrive")
    if root.exists():
        hits = list(root.rglob("amazon-meta.txt"))[:5]
        if hits:
            return str(hits[0])
        hits = list(root.rglob("amazon-meta.txt.gz"))[:5]
        if hits:
            return str(hits[0])
    return None

def ensure_amazon_meta():
    found = find_existing(CANDIDATES)
    if found and found.endswith(".txt") and not found.endswith(".gz"):
        print("Found:", found)
        return found

    gz_path = found if (found and found.endswith(".gz")) else "/content/amazon-meta.txt.gz"
    txt_path = "/content/amazon-meta.txt"
    drive_txt = "/content/drive/MyDrive/amazon-meta.txt"

    if not (found and found.endswith(".gz")):
        url = "https://snap.stanford.edu/data/bigdata/amazon/amazon-meta.txt.gz"
        print("amazon-meta not on Drive. Downloading from SNAP...")
        print(url)
        import urllib.request
        urllib.request.urlretrieve(url, gz_path)
        print("Downloaded:", gz_path, "size=", os.path.getsize(gz_path))

    print("Extracting", gz_path, "->", txt_path)
    with gzip.open(gz_path, "rb") as f_in, open(txt_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

    try:
        shutil.copy2(txt_path, drive_txt)
        print("Copied to Drive:", drive_txt)
        return drive_txt
    except Exception as e:
        print("Could not copy to Drive (ok):", e)
        return txt_path

AMAZON_META_PATH = ensure_amazon_meta()
print("Using AMAZON_META_PATH =", AMAZON_META_PATH)
print("Exists:", os.path.exists(AMAZON_META_PATH), "size=", os.path.getsize(AMAZON_META_PATH))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import shuffle

def parse_amazon_meta(file_path):
    ids, asins, titles, groups, salesranks, similars = [], [], [], [], [], []
    customer_ids, ratings, votes, helpfuls = [], [], [], []
    current_id = current_asin = current_title = current_group = None
    current_salesrank = current_similar = None
    in_review_section = False

    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        for line in file:
            if line.startswith("Id"):
                in_review_section = False
                current_id = current_asin = current_title = current_group = None
                current_salesrank = current_similar = None

            parts = line.split(":", 1)
            key = parts[0].strip() if parts else ""
            value = parts[1].strip() if len(parts) > 1 else ""

            if key == "Id":
                current_id = value
            elif key == "ASIN":
                current_asin = value
            elif key == "title":
                current_title = value
            elif key == "group":
                current_group = value
            elif key == "salesrank":
                current_salesrank = value
            elif key == "similar":
                current_similar = value
            elif key.startswith("reviews"):
                in_review_section = True
            elif in_review_section and "cutomer:" in line.lower():
                # Amazon-meta review line format
                try:
                    tokens = line.strip().split()
                    # expected pieces include cutomer:, rating:, votes:, helpful:
                    cust = None
                    rating = None
                    vote = None
                    helpful = None
                    for i, tok in enumerate(tokens):
                        tl = tok.lower()
                        if tl.startswith("cutomer") or tl.startswith("customer"):
                            cust = tokens[i + 1] if i + 1 < len(tokens) else None
                        elif tl.startswith("rating"):
                            rating = tokens[i + 1] if i + 1 < len(tokens) else None
                        elif tl.startswith("votes"):
                            vote = tokens[i + 1] if i + 1 < len(tokens) else None
                        elif tl.startswith("helpful"):
                            helpful = tokens[i + 1] if i + 1 < len(tokens) else None
                    if cust is None or rating is None:
                        continue
                    ids.append(current_id)
                    asins.append(current_asin)
                    titles.append(current_title)
                    groups.append(current_group)
                    salesranks.append(current_salesrank)
                    similars.append(current_similar)
                    customer_ids.append(cust)
                    ratings.append(rating)
                    votes.append(vote)
                    helpfuls.append(helpful)
                except Exception:
                    continue

    df = pd.DataFrame({
        "Id": ids,
        "ASIN": asins,
        "Title": titles,
        "Group": groups,
        "Salesrank": salesranks,
        "Similar": similars,
        "Customerid": customer_ids,
        "Rating": ratings,
        "Votes": votes,
        "Helpful": helpfuls,
    })
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
    df = df.dropna(subset=["Rating", "Customerid", "ASIN", "Group"])
    return df

print("Parsing amazon-meta (may take several minutes)...")
df_reviews = parse_amazon_meta(AMAZON_META_PATH)
print("rows:", len(df_reviews))
print(df_reviews["Group"].value_counts().head(10))
print("rating min/max/mean:", df_reviews["Rating"].min(), df_reviews["Rating"].max(), df_reviews["Rating"].mean())

In [ ]:
required_groups = ["Book", "Music", "DVD", "Video"]
filtered_df_reviews = df_reviews[df_reviews["Group"].isin(required_groups)].copy()
filtered_df_reviews = filtered_df_reviews.drop_duplicates(subset=["Customerid", "ASIN", "Group", "Rating"])
print("filtered rows:", len(filtered_df_reviews))

def build_pair_matrices(df, cat_a, cat_b, rating_threshold=30):
    user_category_counts = df.groupby(["Customerid", "Group"]).size().reset_index(name="num_ratings")
    filtered_users = user_category_counts[user_category_counts["num_ratings"] >= rating_threshold]
    valid_users = filtered_users[filtered_users["Group"].isin([cat_a, cat_b])]
    valid_users = valid_users.groupby("Customerid").filter(lambda x: x["Group"].nunique() > 1)

    filtered_reviews = df[df["Customerid"].isin(valid_users["Customerid"])]
    filtered_reviews = filtered_reviews[filtered_reviews["Group"].isin([cat_a, cat_b])].copy()
    filtered_reviews["Rating"] = pd.to_numeric(filtered_reviews["Rating"], errors="coerce")

    domain_stats = (
        filtered_reviews.groupby("Group")
        .agg(num_users=("Customerid", "nunique"), num_items=("ASIN", "nunique"), num_ratings=("Rating", "count"))
        .reset_index()
    )
    domain_stats["density_level"] = domain_stats["num_ratings"] / (domain_stats["num_users"] * domain_stats["num_items"])
    source_group = domain_stats.nlargest(1, "density_level")["Group"].values[0]
    target_group = domain_stats.nsmallest(1, "density_level")["Group"].values[0]

    source_matrix = filtered_reviews[filtered_reviews["Group"] == source_group]
    target_matrix = filtered_reviews[filtered_reviews["Group"] == target_group]

    source_rating_matrix = source_matrix.pivot_table(index="Customerid", columns="ASIN", values="Rating", aggfunc="mean").fillna(0)
    target_rating_matrix = target_matrix.pivot_table(index="Customerid", columns="ASIN", values="Rating", aggfunc="mean").fillna(0)

    # align shared users (user-overlap)
    common_users = source_rating_matrix.index.intersection(target_rating_matrix.index)
    source_rating_matrix = source_rating_matrix.loc[common_users]
    target_rating_matrix = target_rating_matrix.loc[common_users]

    meta = {
        "cat_a": cat_a,
        "cat_b": cat_b,
        "source": source_group,
        "target": target_group,
        "n_users": len(common_users),
        "n_items_s": source_rating_matrix.shape[1],
        "n_items_t": target_rating_matrix.shape[1],
        "n_ratings_t": int((target_rating_matrix.to_numpy() > 0).sum()),
        "density_t": float((target_rating_matrix.to_numpy() > 0).sum() / target_rating_matrix.size),
        "mean_rating_t": float(target_rating_matrix.to_numpy()[target_rating_matrix.to_numpy() > 0].mean()),
    }
    return source_rating_matrix, target_rating_matrix, meta

pair_data = {}
for a, b in CATEGORY_PAIRS:
    S, T, meta = build_pair_matrices(filtered_df_reviews, a, b, RATING_THRESHOLD)
    key = f"{meta['source']}->{meta['target']}"
    pair_data[key] = {"S": S, "T": T, "meta": meta}
    print(key, meta)

In [ ]:
def evaluate_mu_baseline(R_true, mu):
    mask = R_true > 0
    if not np.any(mask):
        return np.nan, np.nan
    err = R_true[mask] - mu
    return float(np.mean(np.abs(err))), float(np.sqrt(np.mean(err ** 2)))

def cross_validate_mu_baseline(target_rating_matrix, n_folds=5, random_state=42, fractions=(0.2, 0.4, 0.6, 0.8)):
    R = target_rating_matrix.to_numpy() if hasattr(target_rating_matrix, "to_numpy") else np.asarray(target_rating_matrix)
    pairs = [(u, i, R[u, i]) for u in range(R.shape[0]) for i in range(R.shape[1]) if R[u, i] > 0]
    pairs = shuffle(pairs, random_state=random_state)
    fold_size = len(pairs) // n_folds

    frac_mae = {f: [] for f in fractions}
    frac_rmse = {f: [] for f in fractions}

    for k in range(n_folds):
        te_idx = set(range(k * fold_size, (k + 1) * fold_size if k < n_folds - 1 else len(pairs)))
        te = [pairs[j] for j in sorted(te_idx)]
        tr = [pairs[j] for j in range(len(pairs)) if j not in te_idx]

        R_test = np.zeros_like(R)
        for u, i, r in te:
            R_test[u, i] = r

        idx = shuffle(list(range(len(tr))), random_state=random_state)
        for frac in fractions:
            cut = max(1, int(len(tr) * frac))
            partial = [tr[j] for j in idx[:cut]]
            R_part = np.zeros_like(R)
            for u, i, r in partial:
                R_part[u, i] = r
            mu = R_part[R_part > 0].mean() if np.any(R_part > 0) else 0.0
            mae_p, rmse_p = evaluate_mu_baseline(R_test, mu)
            frac_mae[frac].append(mae_p)
            frac_rmse[frac].append(rmse_p)

    def summarize(maes, rmses):
        m = float(np.mean(maes))
        r = float(np.mean(rmses))
        mci = float(1.96 * np.std(maes, ddof=1) / np.sqrt(len(maes)))
        rci = float(1.96 * np.std(rmses, ddof=1) / np.sqrt(len(rmses)))
        return m, mci, r, rci

    out = {}
    for frac in fractions:
        out[frac] = {
            "mae_folds": frac_mae[frac],
            "rmse_folds": frac_rmse[frac],
            "summary": summarize(frac_mae[frac], frac_rmse[frac]),
        }
    return out

all_results = {}
for key, pack in pair_data.items():
    print("\\n====", key, "====")
    res = cross_validate_mu_baseline(pack["T"], N_FOLDS, RANDOM_STATE, FRACTIONS)
    all_results[key] = res
    for frac, payload in res.items():
        m, mci, r, rci = payload["summary"]
        print(f"mu-baseline TR={int(frac*100)}%: MAE={m:.4f} ± {mci:.4f} | RMSE={r:.4f} ± {rci:.4f}")

In [ ]:
print("=" * 72)
print("COPY EVERYTHING BELOW THIS LINE AND PASTE INTO CURSOR CHAT")
print("=" * 72)
print("TASK: MTAP_Revision_mu_baseline")
print(f"SEED={RANDOM_STATE} FOLDS={N_FOLDS} THRESHOLD={RATING_THRESHOLD}")
print(f"PATH={AMAZON_META_PATH}")
print()

# Paper-style CX labels by source->target if they match manuscript Table
paper_map = {
    "Book->Music": "CX1",
    "DVD->Book": "CX2",
    "Video->Book": "CX3",
    "DVD->Music": "CX4",
    "Video->Music": "CX5",
    "DVD->Video": "CX6",
}

print("PAIR_META")
for key, pack in pair_data.items():
    m = pack["meta"]
    cx = paper_map.get(key, "NA")
    print(f"{cx}\t{key}\tusers={m['n_users']}\titems_t={m['n_items_t']}\tratings_t={m['n_ratings_t']}\tdensity_t={m['density_t']:.6f}\tmean_r={m['mean_rating_t']:.4f}")

print()
print("MU_BASELINE_TABLE")
print("CX\tPair\tTR%\tMAE\tMAE_CI\tRMSE\tRMSE_CI")
for key, res in all_results.items():
    cx = paper_map.get(key, "NA")
    for frac, payload in res.items():
        m, mci, r, rci = payload["summary"]
        print(f"{cx}\t{key}\t{int(frac*100)}\t{m:.4f}\t{mci:.4f}\t{r:.4f}\t{rci:.4f}")

print()
print("MU_BASELINE_FOLDS_MAE_20PCT")
for key, res in all_results.items():
    cx = paper_map.get(key, "NA")
    folds = ",".join(f"{x:.4f}" for x in res[0.2]["mae_folds"])
    print(f"{cx}\t{key}\t{folds}")

print("=" * 72)
print("END_OF_OUTPUT")
print("=" * 72)